# Jurisight – Part 2: Model Training (Colab/Drive)

This notebook fine-tunes a transformer-based model (e.g., Legal-BERT or RoBERTa) to predict legal verdicts.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Install dependencies

Pin versions if required by your project.


In [ ]:
!pip -q install transformers datasets accelerate evaluate scikit-learn torch

## Load splits from Drive


In [ ]:
from pathlib import Path
import pandas as pd

DATA_ROOT = Path('/content/drive/MyDrive/Jurisight/data')
splits_dir = DATA_ROOT / 'splits'
train_df = pd.read_csv(splits_dir / 'train.csv')
val_df = pd.read_csv(splits_dir / 'val.csv')
test_df = pd.read_csv(splits_dir / 'test.csv')
train_df.head()

## Tokenization

Switch the model checkpoint to `nlpaueb/legal-bert-base-uncased` or another legal model as needed.


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

model_checkpoint = 'nlpaueb/legal-bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=512)

train_ds = Dataset.from_pandas(train_df).map(tokenize_batch, batched=True)
val_ds = Dataset.from_pandas(val_df).map(tokenize_batch, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize_batch, batched=True)

## Training configuration


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

num_labels = train_df['label'].nunique()
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels, average='weighted')

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/Jurisight/models/legal-bert',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

## Train and evaluate


In [ ]:
trainer.train()
trainer.evaluate(test_ds)